# 08 · Post-action cash realization
## Feature intervention · complete terminal callback · copied-engine one-step effects

**Question:** are sell orders calculated for the farm actions the coordinated policy actually emits?

This notebook retains the existing rich route menu, assignment objective, worker actions, hiring and terminal reserve rules. It compares market orders based on the base planner's actions against the **same rule** applied after the emitted actions.

**No full games, model fits, installations, downloads or cloud operations.** A maximum 180-second worker budget, 10-second heartbeats, a 500 ms maximum terminal-callback gate and per-episode checkpoints bound the work. Every one-step branch starts from a frozen saved observation. Cash deltas are **not additive season gains** and are not Kaggle ratings.

Read `FEATURE_RESEARCH.md` for the hypothesis, information boundary, controls and remaining research rounds. This is a live notebook: it has no fabricated AWS outputs.

### 1 · Verify the existing kernel and locate this package
Use **Kaggriculture Manual (verified source)**. Do not reinstall or repeat notebooks 05–07.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Markdown, FileLink

BASE = Path.cwd()
if not (BASE / 'run_cash.py').exists():
    BASE = Path.home() / 'kaggriculture_cash_realization'
assert (BASE / 'run_cash.py').is_file(), 'Open this notebook inside the extracted package.'
RESUME = Path.home() / 'kaggriculture_manual_resume'
runtime = json.loads((RESUME / 'state/runtime.json').read_text())
assert Path(sys.executable).absolute() == Path(runtime['executable']).absolute(), 'Select the verified kernel.'
sys.path.insert(0, str(BASE))
OUT = BASE / 'outputs'
print('Package:', BASE)
print('Interpreter:', sys.executable)
print('No setup or previous study is rerun by this notebook.')

Package: /home/sagemaker-user/kaggriculture_cash_realization
Interpreter: /home/sagemaker-user/projects/kaggriculture/.venv/bin/python
No setup or previous study is rerun by this notebook.


### 2 · Start from the verified returned evidence
The earlier deadline-removal test was an independent-worker diagnostic. The frozen full policy already has collection-to-deposit deadlines, task exclusion, optional watering and two-collection routes. It would be misleading to call their reimplementation a new advantage over that baseline.

The new hypothesis concerns an actual integration boundary: replaced farmer/hand actions paired with unchanged orders computed for different actions. Its incidence on the saved episodes is not assumed.

In [2]:
review = json.loads((BASE / 'reference/input_review.json').read_text())
display(pd.DataFrame([
    ('Notebook 07', review['notebook07_status']),
    ('Notebook 06', review['notebook06_status']),
    ('Saved observations', review['saved_observations']),
    ('Worker decisions in each prior probe', review['worker_decisions_per_probe']),
    ('Deadline-removal first-action changes', review['deadline_probe_action_changes']),
    ('Prior extractor max (ms)', review['extractor_ms']['max']),
    ('Verified submission score', 'Not available'),
], columns=['Evidence', 'Result']))

,Evidence,Result
0,Notebook 07,ACTION_FEATURE_AUDIT_PASSED
1,Notebook 06,WORKFORCE_COVERAGE_PASSED
2,Saved observations,161
3,Worker decisions in each prior probe,602
4,Deadline-removal first-action changes,19
5,Prior extractor max (ms),3.132844
6,Verified submission score,Not available


### 3 · Execute the bounded experiment
This runs tests, verifies the prior pass and pinned data/code, checks five synthetic engine fixtures, and replays the 161 final-day observations.

Control outputs must equal the saved source actions. Farm actions and non-SELL orders must be equal between arms. Both complete callbacks are timed. A separate evaluator then runs each one-step branch under the recorded rival action and verifies the control reproduces the recorded next state.

**On failure, stop here and bundle the diagnostics.** Do not raise limits or retry unchanged.

In [3]:
import os
import selectors
import signal
import subprocess
import time

def run_bounded_notebook_stage():
    command = [sys.executable, str(BASE / 'run_cash.py'), 'run', '--seconds', '180']
    proc = subprocess.Popen(command, cwd=BASE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True,
        env={**os.environ, 'PYTHONUNBUFFERED':'1', 'PYTHONDONTWRITEBYTECODE':'1'})
    selector = selectors.DefaultSelector()
    selector.register(proc.stdout, selectors.EVENT_READ)
    started = time.monotonic()
    try:
        while proc.poll() is None:
            if time.monotonic() - started > 210:
                # run_cash owns a separate worker group and an inner hard deadline.
                # Terminating its parent also invokes its worker cleanup handler.
                os.killpg(proc.pid, signal.SIGINT)
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL); proc.wait()
                raise TimeoutError('Notebook emergency deadline. Bundle diagnostics; do not retry unchanged.')
            for key, _ in selector.select(timeout=1):
                line = key.fileobj.readline()
                if line: print(line.rstrip(), flush=True)
        for line in proc.stdout: print(line.rstrip(), flush=True)
        if proc.returncode:
            raise RuntimeError('Notebook 08 stopped. Save and run the bundle command from START_HERE.md.')
    finally:
        selector.close()
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGINT)
            try: proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL); proc.wait()

run_bounded_notebook_stage()

............................................................UPLOAD THIS RESULTS FILE: /tmp/tmp88aullrt/kaggriculture_cash_realization_results.zip
..................
----------------------------------------------------------------------
Ran 78 tests in 1.910s

OK
{"utc": "2026-09-11T23:14:55.860117+00:00", "stage": "PREFLIGHT_PASSED", "verified_objects": 10}
{"utc": "2026-09-11T23:14:55.897893+00:00", "stage": "MECHANICS_PASSED", "fixtures": 5}
{"utc": "2026-09-11T23:14:56.821630+00:00", "stage": "EPISODE_CHECKPOINT_SAVED", "episode": 1, "total": 7, "observations": 23, "market_changes": 0, "max_callback_ms": 4.7938619973137975}
{"utc": "2026-09-11T23:14:58.155708+00:00", "stage": "EPISODE_CHECKPOINT_SAVED", "episode": 2, "total": 7, "observations": 23, "market_changes": 4, "max_callback_ms": 94.43052399728913}
{"utc": "2026-09-11T23:14:59.067601+00:00", "stage": "EPISODE_CHECKPOINT_SAVED", "episode": 3, "total": 7, "observations": 23, "market_changes": 0, "max_callback_ms": 3.6482360010

### 4 · Read the acceptance decision
A pass here is a replay/implementation result, not proof that the agent is stronger. `STOP_NO_ACTIVATION` is a useful stopping result. A positive one-step effect may be only a cash-timing shift that cancels in a later turn.

In [4]:
from run_cash import verify
report = verify()
print('Status:', report['status'])
print('Decision:', report['decision'])
print('Complete terminal callback gate:', report['complete_terminal_callback_gate'])
print('Official metric effect measured:', report['official_metric_effect_measured'])
print('New complete games:', report['new_complete_games'])
display(pd.DataFrame([
    ('Control matches saved actions', report['all_control_actions_match_recorded']),
    ('Same farm actions in both arms', report['same_farm_actions_all_states']),
    ('Same non-SELL / hiring orders', report['same_non_sell_orders_all_states']),
    ('Exact source transitions reproduced', report['recorded_interpreter_transition_parity_checks']),
    ('Changed market actions', report['market_action_changes']),
    ('Positive one-step states', report['positive_one_step_states']),
    ('Negative one-step states', report['negative_one_step_states']),
    ('Candidate descriptors', report['candidate_descriptors']),
], columns=['Gate or observation', 'Result']))

Status: POST_ACTION_CASH_AUDIT_PASSED
Decision: ACTIVATED_REQUIRES_PAIRED_CONTINUATION
Complete terminal callback gate: PASSED_ON_REPLAYED_STATES
Official metric effect measured: False
New complete games: 0


,Gate or observation,Result
0,Control matches saved actions,True
1,Same farm actions in both arms,True
2,Same non-SELL / hiring orders,True
3,Exact source transitions reproduced,161
4,Changed market actions,11
5,Positive one-step states,6
6,Negative one-step states,0
7,Candidate descriptors,158


### 5 · Candidate dictionary and availability
There are **16 fields per product across nine products, plus 14 aggregates**. All `cash.*` fields use the current legal observation and copied effects of our own candidate actions. They do not use the next state, reward, rival inventory or rival action.

`one_step_effects.csv` contains evaluator outcomes separately. Do not feed it into a model without a separately registered, group-isolated training design. Provenance fields such as seed and episode id are not feature columns.

In [5]:
registry = pd.read_csv(OUT / 'feature_registry.csv')
products = pd.read_csv(OUT / 'product_features.csv')
episode_effects = pd.read_csv(OUT / 'effects_by_episode.csv')
display(registry.head(16))
display(episode_effects)
print('Episode rows are descriptive. They share two seed blocks and one opponent, not seven independent experiments.')

,feature,distinct_values,minimum,maximum,nonzero_fraction,status
0,cash.product.CARROT.shed_before,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
1,cash.product.CARROT.shed_after,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
2,cash.product.CARROT.shed_delta,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
3,cash.product.CARROT.control_sell_requested,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
4,cash.product.CARROT.aligned_sell_requested,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
5,cash.product.CARROT.control_sellable_units,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
6,cash.product.CARROT.aligned_sellable_units,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
7,cash.product.CARROT.sellable_units_delta,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
8,cash.product.CARROT.control_residual_units,1,0.0,0.0,0.0,causal_candidate_not_competition_validated
9,cash.product.CARROT.aligned_residual_units,1,0.0,0.0,0.0,causal_candidate_not_competition_validated


,seed,seat,opponent,source_arm,episode_id,observations,positive_one_step_states,negative_one_step_states,mean_one_step_cash_delta,max_one_step_cash_delta,min_one_step_cash_delta
0,1601,0,livestock_fertilizer,coordinated,c3c1b698f20aad8101da,23,2,0,34.782609,714.0,0.0
1,1601,0,livestock_fertilizer,sequential,35cba1b6eb288052fa45,23,0,0,0.000000,0.0,0.0
2,1601,1,livestock_fertilizer,coordinated,2839ed2a525766625395,23,2,0,34.782609,714.0,0.0
3,1601,1,livestock_fertilizer,sequential,8eb5c6031990c805ca5f,23,0,0,0.000000,0.0,0.0
4,1602,0,livestock_fertilizer,coordinated,8fd4963bad01e11bb7c6,23,2,0,64.434783,1440.0,0.0
5,1602,0,livestock_fertilizer,sequential,a17b824709482725f962,23,0,0,0.000000,0.0,0.0
6,1602,1,livestock_fertilizer,sequential,6c5b293d1d6c0e2ac853,23,0,0,0.000000,0.0,0.0


Episode rows are descriptive. They share two seed blocks and one opponent, not seven independent experiments.


### 6 · Prior signal and current inventory coverage
Compare the prior simplified diagnostic with the new post-action representation. Nonzero coverage is not by itself an improvement.

In [6]:
from visualize import charts, save_dashboard
figures = charts(OUT, BASE / 'reference')
figures[0].show()
figures[1].show()

### 7 · Realized one-step cash — not a season score
Each point uses a fresh copy of its saved state. The recorded rival action is held fixed **only in the evaluator**. These points are not a rollout of the treatment policy; adding them across turns would double-count overlapping opportunities.

The last-callback chart is a fixed-state terminal branch, not the outcome of a newly played season.

In [7]:
figures[2].show()
figures[3].show()

### 8 · Complete terminal-callback runtime
Timing includes the new policy's common base planner, route construction, assignment, copied farm effects, feature construction and diagnostics. It excludes the replay harness, file I/O and separate outcome evaluator. Both arms must pass the 500 ms maximum-sample gate. This is not a hosted Kaggle timing certification.

In [8]:
figures[4].show()
display(pd.DataFrame(report['callback_ms']).T)
print(report['timing_scope'])

,max,median,p95
aligned,20.882275,3.810374,16.575163
control,94.430524,3.616195,17.198914


Entire new terminal __call__, including common base, routing, post-action simulation, features and diagnostics. Excludes harness I/O/evaluator; not hosted Kaggle timing. Cache hits retain original timings.


### 9 · Activation, not feature importance
Constant or highly redundant fields can be useful accounting invariants but are not evidence of predictive contribution. A learned scorer has not been fitted. The worker-domain restriction of the old callback also remains; the separate workforce extractor does not remove it from the agent.

In [9]:
figures[5].show()
display(Markdown('**Research remains open.** ' + ' '.join(report['limitations'])))

**Research remains open.** Frozen seven-game development slice; eighth game was latency-censored; two seed blocks, one opponent. Each intervention is a separate one-step branch. Do not sum turn deltas into a season gain. Recorded rival action fixed for each counterfactual; no evidence about adaptive opponent response. Frozen callback still admits at most three hired hands on either farm. Broader-workforce extractor coverage does not certify this callback. One-step evaluator sees both saved private states, but policies/features receive only their own legal observation. Source already contains deadline, water-then-harvest, fertilizer collection and two-collection routes; these are not claimed as newly invented here.

### 10 · Save evidence and stop here
The milestone ends with a decision and saved results. It does not automatically start another experiment.

After all cells finish, **Ctrl+S**, then use the terminal bundle command below. Download the **results ZIP**, not the input ZIP. Stop the JupyterLab application afterwards; do not delete its space.

In [10]:
dashboard = OUT / 'cash_realization.html'
save_dashboard(figures, dashboard)
display(FileLink(str(dashboard)))
print('Saved dashboard:', dashboard)
print('GitHub updated:', report['github_updated'])
print('Next decision:', report['decision'])
print('SAVE NOTEBOOK (Ctrl+S), then run in a JupyterLab terminal:')
import shlex
print('cd ' + shlex.quote(str(BASE)))
print(shlex.quote(sys.executable) + ' run_cash.py bundle')
print('Return: kaggriculture_cash_realization_results.zip')

/home/sagemaker-user/kaggriculture_cash_realization/outputs/cash_realization.html

Saved dashboard: /home/sagemaker-user/kaggriculture_cash_realization/outputs/cash_realization.html
GitHub updated: False
Next decision: ACTIVATED_REQUIRES_PAIRED_CONTINUATION
SAVE NOTEBOOK (Ctrl+S), then run in a JupyterLab terminal:
cd /home/sagemaker-user/kaggriculture_cash_realization
/home/sagemaker-user/projects/kaggriculture/.venv/bin/python run_cash.py bundle
Return: kaggriculture_cash_realization_results.zip


### Sources and follow-on research
See `FEATURE_RESEARCH.md` for pinned source links and the wider research agenda. This family must be followed by a small paired continuation only if it activates and passes its gates. Later rounds remain open: route commitment and alternative-action value; crop time-to-cash; water/fertilizer treatment; livestock feed/care economics; capital allocation; public market history; broad-workforce runtime and grouped opponent stability.

**A score of 3140.0 remains the user-supplied research target. No leaderboard rating is produced by this notebook.**